In [ ]:
import pandas as pd

df = pd.read_csv(
    r"historical_data.csv"
)

df["created_at"] = pd.to_datetime(df["created_at"])
df["actual_delivery_time"] = pd.to_datetime(df["actual_delivery_time"])

df.head()

df["created_at"] = pd.to_datetime(df["created_at"])
df["actual_delivery_time"] = pd.to_datetime(df["actual_delivery_time"])

df["delivery_minutes"] = (
    df["actual_delivery_time"] - df["created_at"]
).dt.total_seconds() / 60

valid = df["delivery_minutes"].dropna()
threshold = 5 * valid.mean()

outliers = df.loc[
    df["delivery_minutes"] > threshold,
    ["created_at", "actual_delivery_time", "delivery_minutes",
     "market_id", "store_id", "total_items"]
].sort_values("delivery_minutes", ascending=False)

print(f"5x-mean threshold: {threshold:.1f} minutes")
print(f"Flagged orders: {len(outliers)}")
display(outliers.head(20))

In [ ]:
for hours in [4, 12, 24]:
    count = (df["delivery_minutes"] > hours * 60).sum()
    print(f"Over {hours} hours: {count} orders")

display(
    outliers.loc[
        outliers["delivery_minutes"] > 24 * 60
    ]
)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv(
    r"historical_data.csv"
)

data["created_at"] = pd.to_datetime(data["created_at"], utc=True, errors="coerce")
data["actual_delivery_time"] = pd.to_datetime(
    data["actual_delivery_time"], utc=True, errors="coerce"
)

# The target is elapsed delivery time in minutes.
data["delivery_minutes"] = (
    data["actual_delivery_time"] - data["created_at"]
).dt.total_seconds() / 60

# A target can't be calculated when either timestamp is missing.
data = data.dropna(subset=["created_at", "delivery_minutes"]).copy()

# Extract order-time features, then sort so testing uses later orders.
local_time = data["created_at"].dt.tz_convert("America/Los_Angeles")
data["order_hour"] = local_time.dt.hour
data["order_day_of_week"] = local_time.dt.dayofweek
data = data.sort_values("created_at")

# Keep timestamps and the target out of the model inputs.
X = data.drop(columns=["created_at", "actual_delivery_time", "delivery_minutes"])
y = data["delivery_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv(
    r"historical_data.csv"
)

data["created_at"] = pd.to_datetime(
    data["created_at"], utc=True, errors="coerce"
)
data["actual_delivery_time"] = pd.to_datetime(
    data["actual_delivery_time"], utc=True, errors="coerce"
)

data["delivery_minutes"] = (
    data["actual_delivery_time"] - data["created_at"]
).dt.total_seconds() / 60

data = data.dropna(
    subset=["created_at", "delivery_minutes"]
).copy()

local_time = data["created_at"].dt.tz_convert("America/Los_Angeles")
data["order_hour"] = local_time.dt.hour
data["order_day_of_week"] = local_time.dt.dayofweek

data = data.sort_values("created_at")

X = data.drop(
    columns=["created_at", "actual_delivery_time", "delivery_minutes"]
)
y = data["delivery_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Exclude high-cardinality store IDs for this first model.
X_train_model = X_train.drop(columns=["store_id"], errors="ignore").copy()
X_test_model = X_test.drop(columns=["store_id"], errors="ignore").copy()

# Treat these IDs as categories, not ordered numeric values.
categorical_columns = [
    col for col in ["market_id", "order_protocol", "store_primary_category"]
    if col in X_train_model.columns
]
for frame in (X_train_model, X_test_model):
    for col in categorical_columns:
        frame[col] = frame[col].map(
            lambda value: str(value) if pd.notna(value) else np.nan
        )

numeric_columns = [
    col for col in X_train_model.columns
    if col not in categorical_columns
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_columns,
        ),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])

model.fit(X_train_model, y_train)
predictions = model.predict(X_test_model)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5

print(f"Test MAE:  {mae:.2f} minutes")
print(f"Test RMSE: {rmse:.2f} minutes")

In [ ]:
baseline_predictions = np.full(len(y_test), y_train.median())

print(f"Baseline MAE:  {mean_absolute_error(y_test, baseline_predictions):.2f} minutes")
print(f"Baseline RMSE: {mean_squared_error(y_test, baseline_predictions) ** 0.5:.2f} minutes")

In [ ]:
errors = pd.DataFrame({
    "actual_minutes": y_test,
    "predicted_minutes": predictions,
})
errors["absolute_error_minutes"] = (
    errors["actual_minutes"] - errors["predicted_minutes"]
).abs()

display(errors.sort_values("absolute_error_minutes", ascending=False).head(10))

In [ ]:
worst = errors.sort_values(
    "absolute_error_minutes", ascending=False
).head(10)

# Show their input features alongside actual and predicted delivery times.
display(pd.concat([worst, X_test.loc[worst.index]], axis=1))

# Show the longest deliveries that the model could have learned from.
display(
    data.loc[X_train.index]
        .nlargest(10, "delivery_minutes")[
            ["created_at", "actual_delivery_time", "delivery_minutes",
             "market_id", "store_id", "total_items"]
        ]
)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv(
    r"historical_data.csv"
)

data["created_at"] = pd.to_datetime(
    data["created_at"], utc=True, errors="coerce"
)
data["actual_delivery_time"] = pd.to_datetime(
    data["actual_delivery_time"], utc=True, errors="coerce"
)

data["delivery_minutes"] = (
    data["actual_delivery_time"] - data["created_at"]
).dt.total_seconds() / 60

# Drop rows without usable timestamps or with nonpositive delivery durations.
data = data.dropna(
    subset=["created_at", "actual_delivery_time", "delivery_minutes"]
).copy()
data = data.loc[data["delivery_minutes"] > 0].copy()

# Define outliers as deliveries longer than 4 hours.
outlier_cutoff_minutes = 4 * 60
outliers = data.loc[data["delivery_minutes"] > outlier_cutoff_minutes].copy()
print(f"Excluded deliveries over 4 hours: {len(outliers)}")

# Keep only the target population for this model.
data = data.loc[data["delivery_minutes"] <= outlier_cutoff_minutes].copy()

# Make order-time features, then sort so the test set contains later orders.
local_time = data["created_at"].dt.tz_convert("America/Los_Angeles")
data["order_hour"] = local_time.dt.hour
data["order_day_of_week"] = local_time.dt.dayofweek
data = data.sort_values("created_at").reset_index(drop=True)

onshift = data["total_onshift_dashers"]
busy = data["total_busy_dashers"]

valid_counts = (
    onshift.gt(0)
    & busy.ge(0)
    & busy.le(onshift)
)

data["busy_dasher_ratio"] = (busy / onshift).where(valid_counts)
X = data.drop(
    columns=["created_at", "actual_delivery_time", "delivery_minutes"]
)
y = data["delivery_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Longest delivery remaining:", f"{y.max():.1f} minutes")

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Leave store_id out of this first model because it has many unique values.
X_train_model = X_train.drop(columns=["store_id"], errors="ignore").copy()
X_test_model = X_test.drop(columns=["store_id"], errors="ignore").copy()

# These are category labels, not numeric quantities.
categorical_columns = [
    col for col in ["market_id", "order_protocol", "store_primary_category"]
    if col in X_train_model.columns
]

# Convert category values to text; preserve missing values for imputation.
for frame in (X_train_model, X_test_model):
    for col in categorical_columns:
        frame[col] = frame[col].map(
            lambda value: str(value) if pd.notna(value) else np.nan
        )

numeric_columns = [
    col for col in X_train_model.columns
    if col not in categorical_columns
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_columns,
        ),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])

model.fit(X_train_model, y_train)
predictions = model.predict(X_test_model)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5

print(f"Test MAE:  {mae:.2f} minutes")
print(f"Test RMSE: {rmse:.2f} minutes")

In [ ]:
baseline_predictions = np.full(len(y_test), y_train.median())

print(
    f"Baseline MAE: "
    f"{mean_absolute_error(y_test, baseline_predictions):.2f} minutes"
)
print(
    f"Baseline RMSE: "
    f"{mean_squared_error(y_test, baseline_predictions) ** 0.5:.2f} minutes"
)

In [ ]:
onshift = data["total_onshift_dashers"]
busy = data["total_busy_dashers"]

# The ratio is meaningful only when the counts are nonnegative
# and busy dashers do not exceed on-shift dashers.
valid_counts = (
    onshift.gt(0)
    & busy.ge(0)
    & busy.le(onshift)
)

data["busy_dasher_ratio"] = (busy / onshift).where(valid_counts)